# Monitor de nuevos productos - Moov
Detecta productos nuevos de Nike y Adidas en moov.com.ar

**Como usarlo:** corre todas las celdas en orden cada vez que quieras revisar.

- Primera vez: guarda el snapshot inicial en tu Drive.
- Las siguientes: muestra solo los productos nuevos.

In [ ]:
# Celda 1: Instalacion (solo la primera vez tarda ~2 min)
!pip install playwright -q
!playwright install chromium
!playwright install-deps chromium

In [ ]:
# Celda 2: Conectar Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Celda 3: Configuracion
import json
import os
from datetime import datetime
from playwright.async_api import async_playwright

URL = "https://www.moov.com.ar/hombre?cgid=hombre&prefn1=brand&prefn2=storeInventoryIds&prefv1=adidas|Nike&prefv2=121|102|200|221|165|049|234|077|148|219|052|188|011|214|228|043|035|232|warehouse&srule=newest-products&start=0&sz=36"
SNAPSHOT_FILE = "/content/drive/MyDrive/moov_snapshot.json"

In [ ]:
# Celda 4: Funcion para obtener productos
JS_EXTRACT = """
() => {
    const results = [];
    const gtmInputs = document.querySelectorAll('.productGtmData');
    gtmInputs.forEach((input) => {
        try {
            const data = JSON.parse(input.value);
            const item = data[0];
            const tile = input.closest('.product');
            let url = "";
            if (tile) {
                const allLinks = tile.querySelectorAll('a[href]');
                for (const link of allLinks) {
                    if (link.href && link.href.includes('moov.com.ar') && !link.href.includes('#')) {
                        url = link.href;
                        break;
                    }
                }
            }
            results.push({
                id: item.item_id || item.id || "",
                name: item.item_name || item.name || "",
                brand: item.item_brand || item.brand || "",
                category: item.item_category || item.category || "",
                price: item.price || "",
                url: url
            });
        } catch(e) {}
    });
    return results;
}
"""

async def get_products():
    async with async_playwright() as p:
        browser = await p.chromium.launch()
        page = await browser.new_page()
        print("Cargando pagina...")
        await page.goto(URL, wait_until='networkidle', timeout=60000)
        try:
            await page.wait_for_selector('.productGtmData', state='attached', timeout=15000)
        except:
            print("Advertencia: tardo en cargar, intentando igual...")
        products = await page.evaluate(JS_EXTRACT)
        await browser.close()
        return products

In [ ]:
# Celda 5: Correr el monitor
products = await get_products()

if not products:
    print("No se pudieron obtener productos. Revisa la URL o intenta de nuevo.")
else:
    print(f"Productos obtenidos: {len(products)}")

    if os.path.exists(SNAPSHOT_FILE):
        with open(SNAPSHOT_FILE) as f:
            previous = json.load(f)
        previous_ids = {p['id'] for p in previous if p.get('id')}
        print(f"Snapshot anterior: {len(previous_ids)} productos")
    else:
        previous_ids = set()
        print("Primera ejecucion: guardando snapshot inicial.")

    new_products = [p for p in products if p.get('id') and p['id'] not in previous_ids]

    print()
    if new_products:
        print(f">>> {len(new_products)} PRODUCTOS NUEVOS:")
        print()
        for p in new_products:
            try:
                price = f"${int(float(p['price'])):,}".replace(',', '.') if p.get('price') else "precio no disponible"
            except:
                price = str(p.get('price', 'precio no disponible'))
            print(f"  [{p['brand']}] {p['name']}")
            print(f"  Precio: {price}")
            if p.get('url'):
                print(f"  Link: {p['url']}")
            print()
    else:
        if previous_ids:
            print("Sin novedades desde la ultima vez.")

    with open(SNAPSHOT_FILE, 'w') as f:
        json.dump(products, f, ensure_ascii=False, indent=2)

    print(f"Revision: {datetime.now().strftime('%d/%m/%Y %H:%M')}")
    print("Snapshot guardado en Google Drive.")